# VisionGate — Model Validation & Evaluation

**University of Ruhuna | EE7204/EC7205 Mini Project**

This notebook quantitatively validates the face-recognition pipeline. It produces the
metrics a biometric attendance system is judged on:

1. **Genuine vs. impostor** embedding-distance distributions.
2. **FAR / FRR** and the **ROC curve + Equal Error Rate (EER)** — used to *justify* the
   recognition threshold instead of guessing it.
3. **Accuracy / precision / recall / F1** with a **confusion matrix**, via stratified
   k-fold cross-validation.
4. A head-to-head **dlib ResNet (face_recognition) vs. OpenCV LBPH** comparison.

All heavy lifting lives in `validate.py`; this notebook just drives it and renders the
results inline.

## 1. Choose the data source

| source | meaning |
|---|---|
| `auto` | local test set if present, else LFW (Olivetti fallback), plus your enrolled face |
| `lfw` | Labeled Faces in the Wild subset (downloads via scikit-learn) |
| `olivetti` | Olivetti faces — tiny, offline-friendly |
| `enrolled` | the faces captured during enrollment |
| `local` | images you drop under `data/test_faces/<name>/` |
| `synthetic` | fabricated data — instant, offline, no dlib (smoke test) |

Start with `synthetic` to confirm everything runs, then switch to `lfw` or `auto` for the
real numbers you put in the report.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import validate

SOURCE = "synthetic"   # <- change to "lfw" or "auto" for real evaluation
N_FOLDS = 5

VAL_DIR = validate.VALIDATION_DIR

## 2. Run the full evaluation

This loads the data, computes embeddings, runs cross-validation for both models, and
saves all charts + a `metrics_summary.json` into `data/validation/`.

In [ ]:
results = validate.run_validation(source=SOURCE, n_splits=N_FOLDS)
summary = results["summary"]

## 3. Headline metrics

In [ ]:
headline = {
    "Data source": summary["data_source"],
    "Images": summary["n_images"],
    "Identities": summary["n_identities"],
    "Genuine mean dist": summary["genuine_mean_distance"],
    "Impostor mean dist": summary["impostor_mean_distance"],
    "EER": f"{summary['EER']:.2%}",
    "EER threshold": summary["EER_threshold"],
    "Config threshold": summary["config_threshold"],
    "FAR @ config": f"{summary['FAR_at_config_threshold']:.2%}",
    "FRR @ config": f"{summary['FRR_at_config_threshold']:.2%}",
}
pd.DataFrame(headline.items(), columns=["metric", "value"])

In [ ]:
pd.DataFrame([summary["dlib"], summary["lbph"]],
             index=["dlib ResNet", "OpenCV LBPH"]).round(3)

## 4. Genuine vs. impostor distance distributions

If the two distributions are well separated, the recogniser is reliable. The dashed line
is the configured decision threshold.

In [ ]:
display(Image(filename=str(VAL_DIR / "distance_distributions.png")))

## 5. Threshold justification — FAR / FRR and ROC / EER

The crossing point of FAR and FRR is the **Equal Error Rate**. Compare its threshold to
`config.RECOGNITION_THRESHOLD` — this is the evidence that the chosen threshold is sound.

In [ ]:
display(Image(filename=str(VAL_DIR / "far_frr.png")))
display(Image(filename=str(VAL_DIR / "roc_curve.png")))

## 6. Confusion matrices (k-fold cross-validation)

In [ ]:
display(Image(filename=str(VAL_DIR / "confusion_dlib.png")))
display(Image(filename=str(VAL_DIR / "confusion_lbph.png")))

## 7. Model comparison — dlib ResNet vs. LBPH

In [ ]:
display(Image(filename=str(VAL_DIR / "model_comparison.png")))

## 8. Threshold sweep table (first & last rows)

In [ ]:
sweep = results["sweep"]
pd.concat([sweep.head(), sweep.tail()]).round(3)

## 9. Raw summary JSON (for the report appendix)

In [ ]:
print(json.dumps({k: v for k, v in summary.items() if k != "charts"}, indent=2))